# VII. Explainability (XAI) — Transformer Classifier

Notebook de explicabilidad del modelo **GLU-Transformer** entrenado sobre datos de interacción del OULAD.

**Estrategia dual:**
1. **Intrínseca** — Extracción de mapas de atención por capa/cabeza para entender el foco temporal del modelo.
2. **Agnóstica** — SHAP (KernelExplainer) para cuantificar la contribución de cada variable (secuencial + estática) a la predicción.
3. **Análisis de errores** — Inspección de los casos más difíciles para el modelo.

> **Split analizado**: Validación  
> **Objetivo**: generar evidencia accionable para intervención educativa temprana.

## 1) Setup, dependencias y configuración

In [1]:
import os
import sys
import json
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix, balanced_accuracy_score,
    roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score,
)

# ─── Estilo visual ────────────────────────────────────────────────────────────
DARK_BG   = '#0F1117'
PANEL_BG  = '#1A1D27'
GRID_CLR  = '#2A2D3A'
TEXT_CLR  = '#E8EAED'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor':   PANEL_BG,
    'text.color':       TEXT_CLR,
    'axes.labelcolor':  TEXT_CLR,
    'xtick.color':      TEXT_CLR,
    'ytick.color':      TEXT_CLR,
    'axes.edgecolor':   GRID_CLR,
    'grid.color':       GRID_CLR,
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'font.family':      'sans-serif',
    'font.sans-serif':  ['DejaVu Sans'],
})
np.set_printoptions(suppress=True, precision=4)

# ─── Proyecto ─────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/workspace/TFM_education_ai_analytics')
os.chdir(PROJECT_ROOT)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL EXPERIMENTO  (✏️ editar aquí)
# ═══════════════════════════════════════════════════════════════════════════════
UPTO_WEEK    = 5
SPLIT        = 'validation'
NUM_CLASSES  = 2
BINARY_MODE  = 'paper'          # paper | original | success_vs_risk
WITH_STATIC  = True

# Derivar rutas automáticamente
TARGET_TAG   = f'{NUM_CLASSES}clases_{BINARY_MODE}' if NUM_CLASSES == 2 else f'{NUM_CLASSES}clases'
DATA_DIR     = PROJECT_ROOT / 'data' / '6_transformer_features'
MODEL_PATH   = PROJECT_ROOT / 'models' / 'transformers' / f'transformer_uptoW{UPTO_WEEK}_{TARGET_TAG}.keras'
HISTORY_PATH = PROJECT_ROOT / 'reports' / 'transformer_training' / f'week_{UPTO_WEEK}' / f'experiments_history_{TARGET_TAG}.json'
REPORT_DIR   = PROJECT_ROOT / 'reports' / 'transformer_training' / f'week_{UPTO_WEEK}'
HARDEST_CSV  = REPORT_DIR / f'hardest_val_examples_uptoW{UPTO_WEEK}_{TARGET_TAG}.csv'

print(f'Configuración:')
print(f'  Semana          : {UPTO_WEEK}')
print(f'  Split           : {SPLIT}')
print(f'  Clases          : {NUM_CLASSES} ({BINARY_MODE})')
print(f'  Con estáticas   : {WITH_STATIC}')
print(f'  Modelo          : {MODEL_PATH.name} (existe: {MODEL_PATH.exists()})')
print(f'  Historial       : {HISTORY_PATH.name} (existe: {HISTORY_PATH.exists()})')
print(f'  Hardest CSV     : {HARDEST_CSV.name} (existe: {HARDEST_CSV.exists()})')
print(f'  TensorFlow      : {tf.__version__}')

2026-03-02 17:55:37.492625: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr

Configuración:
  Semana          : 5
  Split           : validation
  Clases          : 2 (paper)
  Con estáticas   : True
  Modelo          : transformer_uptoW5_2clases_paper.keras (existe: True)
  Historial       : experiments_history_2clases_paper.json (existe: True)
  Hardest CSV     : hardest_val_examples_uptoW5_2clases_paper.csv (existe: True)
  TensorFlow      : 2.20.0


## 2) Carga de datos, modelo y utilidades

In [ ]:
# ─── Importar la arquitectura del Transformer ─────────────────────────────────
mod_path = PROJECT_ROOT / 'educational_ai_analytics' / '2_modeling' / 'transformers' / 'transformer_GLU_classifier.py'
spec = importlib.util.spec_from_file_location('transformer_GLU_classifier', mod_path)
tg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tg)

GLUTransformerClassifier = tg.GLUTransformerClassifier
TransformerEncoderBlock  = tg.TransformerEncoderBlock
GLULayer                 = tg.GLULayer


# ─── Funciones auxiliares ──────────────────────────────────────────────────────
def load_npz_split(split: str, upto_week: int, with_static: bool = True):
    fp = DATA_DIR / split / f'transformer_uptoW{upto_week}.npz'
    d  = np.load(fp, allow_pickle=True)

    X_seq         = d['X_seq'].astype(np.float32)
    mask_pad      = (d['mask_pad'] if 'mask_pad' in d.files else d['mask']).astype(np.int32)
    mask_activity = (d['mask_activity'] if 'mask_activity' in d.files else mask_pad).astype(np.int32)
    y             = d['y'].astype(np.int64)
    ids           = d['ids'].astype(str)

    X_static = d['X_static'].astype(np.float32) if with_static else None

    static_feature_names = d['static_feature_names'] if 'static_feature_names' in d.files else np.array([], dtype=object)
    static_feature_sources = d['static_feature_sources'] if 'static_feature_sources' in d.files else np.array([], dtype=object)
    activities = d['activities'] if 'activities' in d.files else np.array([], dtype=object)

    return {
        'X_seq': X_seq, 'mask_pad': mask_pad, 'mask_activity': mask_activity,
        'y': y, 'ids': ids, 'X_static': X_static,
        'static_feature_names': static_feature_names,
        'static_feature_sources': static_feature_sources,
        'activities': activities,
    }


def filter_classes_binary(payload, binary_mode: str):
    """Aplica el mismo filtro de clases que train_transformer.py."""
    y = payload['y']

    if binary_mode == 'paper':
        # Paper: Pass/Dist (y>=2) -> 0, Withdrawn (y==0) -> 1; Fail (y==1) se descarta
        keep = y != 1
        y_bin = np.where(y[keep] == 0, 1, 0).astype(np.int64)
    elif binary_mode == 'original':
        # Original: same but keep Fail(0)->1, remove Withdrawn(1)
        keep = y != 0
        y_kept = y[keep]
        y_bin = np.where(y_kept == 1, 1, 0).astype(np.int64)
    elif binary_mode == 'success_vs_risk':
        # Success vs Risk: Pass/Dist (>=2) -> 0, Fail/Withdrawn (0,1) -> 1
        keep = np.ones(len(y), dtype=bool)
        y_bin = np.where(y < 2, 1, 0).astype(np.int64)
    else:
        raise ValueError(f'binary_mode desconocido: {binary_mode}')

    result = {}
    for k, v in payload.items():
        if isinstance(v, np.ndarray) and v.shape[0] == len(y):
            result[k] = v[keep]
        else:
            result[k] = v
    result['y'] = y_bin
    return result


def load_model_from_history(model_path, history_path, upto_week, x_seq_sample, x_static_sample):
    """Carga robusta: reconstruye arquitectura desde historial + carga pesos."""
    hist = json.loads(history_path.read_text(encoding='utf-8'))
    candidates = [r for r in hist if int(r.get('hyperparameters', {}).get('upto_week', -1)) == int(upto_week)]
    if not candidates:
        raise ValueError(f'No hay entradas en historial para upto_week={upto_week}')
    hp = candidates[-1]['hyperparameters']

    model = GLUTransformerClassifier(
        latent_d=int(hp.get('latent_d', 256)),
        num_heads=int(hp.get('num_heads', 8)),
        ff_dim=int(hp.get('ff_dim', 128)),
        dropout=float(hp.get('dropout', 0.3)),
        num_classes=int(hp.get('num_classes', NUM_CLASSES)),
        num_layers=int(hp.get('num_layers', 2)),
        max_len=int(x_seq_sample.shape[1]),
        with_static_features=bool(hp.get('with_static', True)),
        static_hidden=tuple(hp.get('static_hidden', [64, 32])),
        head_hidden=tuple(hp.get('head_hidden', [256, 128])),
    )

    # Forward pass de calentamiento para construir pesos
    seq = x_seq_sample[:2].astype(np.float32)
    m = np.ones(seq.shape[:2], dtype=np.int32)
    inputs = [seq, m, m]
    if bool(hp.get('with_static', True)) and x_static_sample is not None:
        inputs.append(x_static_sample[:2].astype(np.float32))
    _ = model(inputs, training=False)

    model.load_weights(model_path)
    return model, hp


# ─── Cargar datos ─────────────────────────────────────────────────────────────
payload_raw = load_npz_split(SPLIT, UPTO_WEEK, with_static=WITH_STATIC)

if NUM_CLASSES == 2:
    payload = filter_classes_binary(payload_raw, BINARY_MODE)
else:
    payload = payload_raw  # 3 o 4 clases: sin filtro

X_seq         = payload['X_seq']
mask_pad      = payload['mask_pad']
mask_activity = payload['mask_activity']
X_static      = payload['X_static']
y             = payload['y']
ids           = payload['ids']
activities    = payload['activities']
static_names  = payload['static_feature_names']

N, W, F = X_seq.shape
print(f'\n📊 Datos cargados ({SPLIT}):')
print(f'  X_seq:     {X_seq.shape}')
print(f'  X_static:  {X_static.shape if X_static is not None else "N/A"}')
print(f'  y:         {y.shape} | distribución: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'  IDs:       {ids.shape}')
print(f'  Semanas:   {W} | Features seq: {F} | Features estáticas: {X_static.shape[1] if X_static is not None else 0}')
print(f'  Actividades: {list(activities.astype(str)[:10])}...'  if len(activities) > 10 else f'  Actividades: {list(activities.astype(str))}')

# ─── Cargar modelo ─────────────────────────────────────────────────────────────
model, hp_loaded = load_model_from_history(
    model_path=MODEL_PATH,
    history_path=HISTORY_PATH,
    upto_week=UPTO_WEEK,
    x_seq_sample=X_seq,
    x_static_sample=X_static,
)
print(f'\n✅ Modelo cargado: {MODEL_PATH.name}')
print(f'  Hiperparámetros: latent_d={hp_loaded.get("latent_d")}, num_heads={hp_loaded.get("num_heads")}, '
      f'ff_dim={hp_loaded.get("ff_dim")}, num_layers={hp_loaded.get("num_layers")}, dropout={hp_loaded.get("dropout")}')

# ─── Predicción rápida para verificar alineación ──────────────────────────────
inputs_val = [X_seq, mask_pad.astype(np.int32), mask_activity.astype(np.int32)]
if X_static is not None:
    inputs_val.append(X_static)

probs = model.predict(inputs_val, verbose=0)
y_pred = np.argmax(probs, axis=1)

bal_acc = balanced_accuracy_score(y, y_pred)
print(f'\n📈 Verificación rápida sobre {SPLIT}:')
print(f'  Balanced Accuracy: {bal_acc:.4f}')
print(f'  Confusion Matrix:\n{confusion_matrix(y, y_pred)}')
print(f'\n{classification_report(y, y_pred, digits=4)}')

if NUM_CLASSES == 2:
    auc_score = roc_auc_score(y, probs[:, 1])
    print(f'  AUC: {auc_score:.4f}')